## 1. Подготовка пайпланов для обработки признаков

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (cross_val_score, train_test_split,
                                     GridSearchCV, StratifiedKFold, RandomizedSearchCV)


In [2]:
load_path = os.path.join('..', 'data', 'processed', 'eda_telco_customer_churn_train.csv')

df = pd.read_csv(load_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4225 entries, 0 to 4224
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                4225 non-null   int64  
 1   Avg Monthly GB Download            4225 non-null   int64  
 2   Avg Monthly Long Distance Charges  4225 non-null   float64
 3   CLTV                               4225 non-null   int64  
 4   Contract                           4225 non-null   object 
 5   Device Protection Plan             4225 non-null   int64  
 6   Gender                             4225 non-null   object 
 7   Internet Service                   4225 non-null   int64  
 8   Multiple Lines                     4225 non-null   int64  
 9   Number of Dependents               4225 non-null   int64  
 10  Offer                              4225 non-null   int64  
 11  Online Backup                      4225 non-null   int64

In [3]:
y = df['Churn']
X_train = df.drop(columns=['Churn'])

In [4]:
numeric = ['Age', 'Avg Monthly GB Download', 'Avg Monthly Long Distance Charges', 
           'CLTV', 'Satisfaction Score', 'Tenure in Months', 'Total Long Distance Charges']
print(f'Всего {len(numeric)} числовых признаков')

Всего 7 числовых признаков


In [5]:
categorical = ['Contract', 'Device Protection Plan', 'Gender', 'Internet Service', 'Multiple Lines', 'Offer', 
              'Online Backup', 'Online Security', 'Paperless Billing', 'Partner',
              'Payment Method', 'Premium Tech Support', 'Streaming TV', 'Total Refunds',
              'Number of Dependents', 'Total Extra Data Charges']
print(f'Всего {len(categorical)} категориальных признаков')

Всего 16 категориальных признаков


In [6]:
num_transformer = Pipeline([
    ('scaler', StandardScaler())
])
cat_transformer = Pipeline([
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numeric),
        ('cat', cat_transformer, categorical)
    ])

## 2. Random Forest

В качестве baseline-модели возьмём случайный лес. Он не требует масштабирования, хорошо работает с дисбалансом классов и устойчив к выбросам.

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('RF', RandomForestClassifier(random_state=42))
])

rf_param_grid = {
    'RF__n_estimators': [50, 100],  
    'RF__max_depth': [None, 10, 20],
    'RF__min_samples_split': [2, 5],  
    'RF__min_samples_leaf': [1, 4],  
    'RF__max_features': ['sqrt', 'log2'], 
    'RF__class_weight': [None, 'balanced']
}

rf_cv = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1
)

rf_cv.fit(X_train, y)
print('Random Forest:')
print('Лучшие параметры:', rf_cv.best_params_)
print('Лучшая метрика f1_weighted:', rf_cv.best_score_)


Random Forest:
Лучшие параметры: {'RF__class_weight': 'balanced', 'RF__max_depth': 20, 'RF__max_features': 'sqrt', 'RF__min_samples_leaf': 1, 'RF__min_samples_split': 2, 'RF__n_estimators': 50}
Лучшая метрика f1_weighted: 0.9579646850733126
